# Reviewed deepfake pipeline: real-data PyTorch training
This notebook embeds the current reviewed source. Select a **T4 GPU**, then run cells
in order. Checkpoints and reports go to `MyDrive/deepfake_project`.

The first run uses the existing project's ASVspoof balanced-subset recipe: up to
**2,500 samples per class per official split**, seed 42. Its scores are subset results,
not a full ASVspoof benchmark. Face/audio/fusion training also runs if FakeAVCeleb
processed data or an approved archive is available in the project folder.
No credentials are embedded. Drive mounting may require your authorization.

In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU before running."
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
SOURCE_SHA256 = '3a53cf5154dc675eb36409708d1ce66a7654590c982ea90b5b6fa013c7648125'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256
CODE = Path("/content") / ("DeepFake_reviewed_" + SOURCE_SHA256[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        assert (CODE / name).resolve().is_relative_to(CODE.resolve())
    archive.extractall(CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(CODE / "src"))
print("Reviewed source installed:", SOURCE_SHA256)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/deepfake_project")
DATA.mkdir(parents=True, exist_ok=True)
os.environ["DEEPFAKE_DATA_ROOT"] = str(DATA)
os.environ["DEEPFAKE_RAW_ROOT"] = "/content/raw"
from deepfake.config import load_config, get_path
cfg = load_config()
PER_CLASS = 2500
AUDIO_RUN = "audio_asvspoof_" + SOURCE_SHA256[:8]
logs = DATA / "outputs" / ("training_" + SOURCE_SHA256[:8])
logs.mkdir(parents=True, exist_ok=True)
def run_module(module, *args):
    command = [sys.executable, "-u", "-m", module, *map(str, args)]
    print("Running:", " ".join(command), flush=True)
    with (logs / (module.rsplit(".", 1)[-1] + ".log")).open("a") as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        status = proc.wait()
    if status:
        raise RuntimeError(f"{module} failed (exit {status}); see {logs}")
(logs / "source.json").write_text(json.dumps({"source_sha256": SOURCE_SHA256,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "asvspoof_per_class_per_split": PER_CLASS, "seed": cfg["seed"]}, indent=2))
run_module("deepfake.data.download", "status")

In [ ]:
patch = base64.b64decode("IiIiRG93bmxvYWQsIGV4dHJhY3QgYW5kIHZlcmlmeSB0aGUgcHJvamVjdCBkYXRhc2V0cy4KCkRhdGFzZXRzIChzYXZlZCB1bmRlciA8ZGF0YV9yb290Pi9yYXcvPG5hbWU+Lyk6CiAgZGZkYyAgICAgICAgIERGREMgdHJhaW4gc2FtcGxlIGZyb20gS2FnZ2xlICh+NDAwIHZpZGVvcyB3aXRoIGF1ZGlvKS4gIE5lZWRzIGEgS2FnZ2xlCiAgICAgICAgICAgICAgIGFjY291bnQsIGFjY2VwdGVkIGNvbXBldGl0aW9uIHJ1bGVzLCBhbmQgYW4gQVBJIHRva2VuIChrYWdnbGUuanNvbikuCiAgYXN2c3Bvb2YgICAgIEFTVnNwb29mIDIwMTkgTG9naWNhbCBBY2Nlc3MgKExBLnppcCwgNy42IEdCKS4gRnJlZSwgbm8gYWNjb3VudC4KICBmYWtlYXZjZWxlYiAgRmFrZUFWQ2VsZWIgdjEuMi4gQWNjZXNzIGJ5IHJlcXVlc3QgZm9ybTsgcGFzcyB0aGUgbGluayB5b3UgcmVjZWl2ZQogICAgICAgICAgICAgICB3aXRoIC0tdXJsLCBvciBhbiBhbHJlYWR5IGRvd25sb2FkZWQgYXJjaGl2ZSB3aXRoIC0tYXJjaGl2ZS4KICBjZWxlYmRmICAgICAgQ2VsZWItREYgdjIuIEFjY2VzcyBieSByZXF1ZXN0IGZvcm07IHNhbWUgLS11cmwgLyAtLWFyY2hpdmUgb3B0aW9ucy4KClVzYWdlOgogIHB5dGhvbiAtbSBkZWVwZmFrZS5kYXRhLmRvd25sb2FkIHN0YXR1cwogIHB5dGhvbiAtbSBkZWVwZmFrZS5kYXRhLmRvd25sb2FkIGFzdnNwb29mCiAgcHl0aG9uIC1tIGRlZXBmYWtlLmRhdGEuZG93bmxvYWQgYXN2c3Bvb2YgLS1jaGVjayAgICAgICAgIyBvbmx5IHRlc3QgdGhlIGxpbmsKICBweXRob24gLW0gZGVlcGZha2UuZGF0YS5kb3dubG9hZCBkZmRjCiAgcHl0aG9uIC1tIGRlZXBmYWtlLmRhdGEuZG93bmxvYWQgZmFrZWF2Y2VsZWIgLS11cmwgIjxsaW5rIGZyb20gdGhlIGVtYWlsPiIKICBweXRob24gLW0gZGVlcGZha2UuZGF0YS5kb3dubG9hZCBjZWxlYmRmIC0tYXJjaGl2ZSAvcGF0aC90by9DZWxlYi1ERi12Mi56aXAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0YXJmaWxlCmltcG9ydCB6aXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHJlcXVlc3RzCmZyb20gdHFkbSBpbXBvcnQgdHFkbQoKZnJvbSBkZWVwZmFrZS5jb25maWcgaW1wb3J0IGdldF9wYXRoLCBsb2FkX2NvbmZpZwoKQVNWU1BPT0ZfTEFfVVJMID0gKAogICAgImh0dHBzOi8vZGF0YXNoYXJlLmVkLmFjLnVrL3NlcnZlci9hcGkvY29yZS9iaXRzdHJlYW1zLyIKICAgICJhOWY4N2MzNS1mMDU1LTQwMTUtODBlMi0yZmRmZjBkNDYyNjkvY29udGVudCIKKQpBU1ZTUE9PRl9MQV9TSVpFID0gN182NDBfOTUyXzUyMApBU1ZTUE9PRl9MQV9NSVJST1IgPSAiaHR0cHM6Ly96ZW5vZG8ub3JnL2FwaS9yZWNvcmRzLzY5MDYzMDYvZmlsZXMvTEEuemlwL2NvbnRlbnQiCkFTVlNQT09GX0xBX01ENSA9ICIzMGM5OGYxMWQ4YjJiYzIxZjJjMjU3YmZkNzhiYjVjNSIKVVNFUl9BR0VOVCA9ICJNb3ppbGxhLzUuMCAoZGVlcGZha2UtcmVzZWFyY2gtZG93bmxvYWQpIgoKRkFLRUFWQ0VMRUJfRk9STSA9ICgKICAgICJodHRwczovL2RvY3MuZ29vZ2xlLmNvbS9mb3Jtcy9kL2UvMUZBSXBRTFNmUERkM29WMGF1cW1tV0VnQ1NhVEVRNkNHcEZlQi1velFKMzV4LUJfMFhqZDkzYncvdmlld2Zvcm0iCikKQ0VMRUJERl9SRVBPID0gImh0dHBzOi8vZ2l0aHViLmNvbS95dWV6dW5saS9jZWxlYi1kZWVwZmFrZWZvcmVuc2ljcyIKRkFLRUFWQ0VMRUJfQ0FURUdPUklFUyA9ICgKICAgICJSZWFsVmlkZW8tUmVhbEF1ZGlvIiwKICAgICJSZWFsVmlkZW8tRmFrZUF1ZGlvIiwKICAgICJGYWtlVmlkZW8tUmVhbEF1ZGlvIiwKICAgICJGYWtlVmlkZW8tRmFrZUF1ZGlvIiwKKQpDRUxFQkRGX0ZPTERFUlMgPSAoIkNlbGViLXJlYWwiLCAiQ2VsZWItc3ludGhlc2lzIiwgIllvdVR1YmUtcmVhbCIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gaGVscGVycwoKZGVmIGh0dHBfZG93bmxvYWQodXJsOiBzdHIsIGRlc3Q6IFBhdGgsIGV4cGVjdGVkX3NpemU6IGludCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoOgogICAgIiIiU3RyZWFtIGEgZmlsZSB0byBkaXNrIHdpdGggcmVzdW1lIHN1cHBvcnQgKHNhZmUgdG8gcmUtcnVuIGFmdGVyIGEgQ29sYWIgZGlzY29ubmVjdCkuIiIiCiAgICBkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkb25lID0gZGVzdC5zdGF0KCkuc3Rfc2l6ZSBpZiBkZXN0LmV4aXN0cygpIGVsc2UgMAogICAgaWYgZXhwZWN0ZWRfc2l6ZSBhbmQgZG9uZSA9PSBleHBlY3RlZF9zaXplOgogICAgICAgIHByaW50KGYiQWxyZWFkeSBkb3dubG9hZGVkOiB7ZGVzdH0iKQogICAgICAgIHJldHVybiBkZXN0CiAgICBoZWFkZXJzID0geyJVc2VyLUFnZW50IjogVVNFUl9BR0VOVH0KICAgIGlmIGRvbmU6CiAgICAgICAgaGVhZGVyc1siUmFuZ2UiXSA9IGYiYnl0ZXM9e2RvbmV9LSIKICAgIHdpdGggcmVxdWVzdHMuZ2V0KHVybCwgaGVhZGVycz1oZWFkZXJzLCBzdHJlYW09VHJ1ZSwgdGltZW91dD02MCkgYXMgcjoKICAgICAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgICAgIGlmIGRvbmUgYW5kIHIuc3RhdHVzX2NvZGUgIT0gMjA2OiAgIyBzZXJ2ZXIgaWdub3JlZCBSYW5nZTogc3RhcnQgb3ZlcgogICAgICAgICAgICBkb25lID0gMAogICAgICAgIHRvdGFsID0gZG9uZSArIGludChyLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsIDApKSBvciBleHBlY3RlZF9zaXplCiAgICAgICAgd2l0aCBvcGVuKGRlc3QsICJhYiIgaWYgZG9uZSBlbHNlICJ3YiIpIGFzIGYsIHRxZG0oCiAgICAgICAgICAgIHRvdGFsPXRvdGFsLCBpbml0aWFsPWRvbmUsIHVuaXQ9IkIiLCB1bml0X3NjYWxlPVRydWUsIGRlc2M9ZGVzdC5uYW1lCiAgICAgICAgKSBhcyBiYXI6CiAgICAgICAgICAgIGZvciBjaHVuayBpbiByLml0ZXJfY29udGVudChjaHVua19zaXplPTEgPDwgMjApOgogICAgICAgICAgICAgICAgZi53cml0ZShjaHVuaykKICAgICAgICAgICAgICAgIGJhci51cGRhdGUobGVuKGNodW5rKSkKICAgIGlmIGV4cGVjdGVkX3NpemUgYW5kIGRlc3Quc3RhdCgpLnN0X3NpemUgIT0gZXhwZWN0ZWRfc2l6ZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYie2Rlc3R9IGlzIHtkZXN0LnN0YXQoKS5zdF9zaXplfSBieXRlcywgZXhwZWN0ZWQge2V4cGVjdGVkX3NpemV9LiBSZS1ydW4gdG8gcmVzdW1lLiIKICAgICAgICApCiAgICByZXR1cm4gZGVzdAoKCmRlZiBkb3dubG9hZF91cmwodXJsOiBzdHIsIGRlc3RfZGlyOiBQYXRoKSAtPiBQYXRoOgogICAgIiIiRG93bmxvYWQgYSBsaW5rIGZyb20gYSBkYXRhc2V0IGVtYWlsOiBHb29nbGUgRHJpdmUgbGlua3MgdmlhIGdkb3duLCBvdGhlcnMgdmlhIEhUVFAuIiIiCiAgICBkZXN0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiAiZHJpdmUuZ29vZ2xlLmNvbSIgaW4gdXJsIG9yICJkb2NzLmdvb2dsZS5jb20iIGluIHVybDoKICAgICAgICBpbXBvcnQgZ2Rvd24KCiAgICAgICAgaWYgIi9mb2xkZXJzLyIgaW4gdXJsOgogICAgICAgICAgICBnZG93bi5kb3dubG9hZF9mb2xkZXIodXJsLCBvdXRwdXQ9c3RyKGRlc3RfZGlyKSwgcXVpZXQ9RmFsc2UpCiAgICAgICAgICAgIHJldHVybiBkZXN0X2RpcgogICAgICAgIG91dCA9IGdkb3duLmRvd25sb2FkKHVybCwgb3V0cHV0PXN0cihkZXN0X2RpcikgKyAiLyIsIHF1aWV0PUZhbHNlLCBmdXp6eT1UcnVlKQogICAgICAgIHJldHVybiBQYXRoKG91dCkKICAgIG5hbWUgPSB1cmwuc3BsaXQoIj8iKVswXS5yc3RyaXAoIi8iKS5zcGxpdCgiLyIpWy0xXSBvciAiZG93bmxvYWQuemlwIgogICAgcmV0dXJuIGh0dHBfZG93bmxvYWQodXJsLCBkZXN0X2RpciAvIG5hbWUpCgoKZGVmIF9zYWZlX3RhcmdldChkZXN0OiBQYXRoLCBtZW1iZXI6IHN0cikgLT4gUGF0aDoKICAgIHRhcmdldCA9IChkZXN0IC8gbWVtYmVyKS5yZXNvbHZlKCkKICAgIGlmIG5vdCB0YXJnZXQuaXNfcmVsYXRpdmVfdG8oZGVzdC5yZXNvbHZlKCkpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlVuc2FmZSBwYXRoIGluIGFyY2hpdmU6IHttZW1iZXJ9IikKICAgIHJldHVybiB0YXJnZXQKCgpkZWYgZXh0cmFjdChhcmNoaXZlOiBQYXRoLCBkZXN0OiBQYXRoLCBvbmx5X3ByZWZpeDogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6CiAgICAiIiJFeHRyYWN0IGEgemlwL3RhciBhcmNoaXZlLCBvcHRpb25hbGx5IG9ubHkgbWVtYmVycyB1bmRlciBgb25seV9wcmVmaXhgLiIiIgogICAgZGVzdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwcmludChmIkV4dHJhY3Rpbmcge2FyY2hpdmUubmFtZX0gLT4ge2Rlc3R9IikKICAgIGlmIHppcGZpbGUuaXNfemlwZmlsZShhcmNoaXZlKToKICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShhcmNoaXZlKSBhcyB6ZjoKICAgICAgICAgICAgbWVtYmVycyA9IFttIGZvciBtIGluIHpmLm5hbWVsaXN0KCkgaWYgbm90IG9ubHlfcHJlZml4IG9yIG0uc3RhcnRzd2l0aChvbmx5X3ByZWZpeCldCiAgICAgICAgICAgIGZvciBtIGluIHRxZG0obWVtYmVycywgdW5pdD0iZmlsZSIpOgogICAgICAgICAgICAgICAgX3NhZmVfdGFyZ2V0KGRlc3QsIG0pCiAgICAgICAgICAgICAgICB6Zi5leHRyYWN0KG0sIGRlc3QpCiAgICBlbGlmIHRhcmZpbGUuaXNfdGFyZmlsZShhcmNoaXZlKToKICAgICAgICB3aXRoIHRhcmZpbGUub3BlbihhcmNoaXZlKSBhcyB0ZjoKICAgICAgICAgICAgbWVtYmVycyA9IFsKICAgICAgICAgICAgICAgIG0gZm9yIG0gaW4gdGYuZ2V0bWVtYmVycygpIGlmIG5vdCBvbmx5X3ByZWZpeCBvciBtLm5hbWUuc3RhcnRzd2l0aChvbmx5X3ByZWZpeCkKICAgICAgICAgICAgXQogICAgICAgICAgICBmb3IgbSBpbiB0cWRtKG1lbWJlcnMsIHVuaXQ9ImZpbGUiKToKICAgICAgICAgICAgICAgIF9zYWZlX3RhcmdldChkZXN0LCBtLm5hbWUpCiAgICAgICAgICAgICAgICBpZiBub3QgKG0uaXNmaWxlKCkgb3IgbS5pc2RpcigpKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJVbnNhZmUgYXJjaGl2ZSBtZW1iZXIgKGxpbmsgb3Igc3BlY2lhbCBmaWxlKToge20ubmFtZX0iKQogICAgICAgICAgICAgICAgdGYuZXh0cmFjdChtLCBkZXN0KQogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTm90IGEgemlwIG9yIHRhciBhcmNoaXZlOiB7YXJjaGl2ZX0iKQoKCmRlZiBmaW5kX2Rpcl93aXRoKHJvb3Q6IFBhdGgsIG5hbWVzOiB0dXBsZVtzdHIsIC4uLl0sIG1heF9kZXB0aDogaW50ID0gNCkgLT4gUGF0aCB8IE5vbmU6CiAgICAiIiJGaW5kIHRoZSBmaXJzdCBkaXJlY3RvcnkgdW5kZXIgcm9vdCB0aGF0IGNvbnRhaW5zIGFsbCBvZiBgbmFtZXNgLiIiIgogICAgaWYgbm90IHJvb3QuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHF1ZXVlID0gWyhyb290LCAwKV0KICAgIHdoaWxlIHF1ZXVlOgogICAgICAgIGQsIGRlcHRoID0gcXVldWUucG9wKDApCiAgICAgICAgaWYgYWxsKChkIC8gbikuZXhpc3RzKCkgZm9yIG4gaW4gbmFtZXMpOgogICAgICAgICAgICByZXR1cm4gZAogICAgICAgIGlmIGRlcHRoIDwgbWF4X2RlcHRoOgogICAgICAgICAgICBxdWV1ZSArPSBbKGMsIGRlcHRoICsgMSkgZm9yIGMgaW4gc29ydGVkKGQuaXRlcmRpcigpKSBpZiBjLmlzX2RpcigpXQogICAgcmV0dXJuIE5vbmUKCgpkZWYgY291bnQocm9vdDogUGF0aCwgcGF0dGVybjogc3RyKSAtPiBpbnQ6CiAgICByZXR1cm4gc3VtKDEgZm9yIF8gaW4gcm9vdC5yZ2xvYihwYXR0ZXJuKSkKCgpkZWYgX2luZ2VzdChyYXc6IFBhdGgsIHVybDogc3RyIHwgTm9uZSwgYXJjaGl2ZTogc3RyIHwgTm9uZSkgLT4gTm9uZToKICAgICIiIlNoYXJlZCAtLXVybCAvIC0tYXJjaGl2ZSBoYW5kbGluZyBmb3IgdGhlIHJlcXVlc3QtZm9ybSBkYXRhc2V0cy4iIiIKICAgIGlmIHVybDoKICAgICAgICBnb3QgPSBkb3dubG9hZF91cmwodXJsLCByYXcgLyAiX2Rvd25sb2FkcyIpCiAgICAgICAgYXJjaGl2ZXMgPSBbZ290XSBpZiBnb3QuaXNfZmlsZSgpIGVsc2Ugc29ydGVkKHAgZm9yIHAgaW4gZ290LnJnbG9iKCIqIikgaWYgcC5pc19maWxlKCkpCiAgICBlbHNlOgogICAgICAgIGFyY2hpdmVzID0gW1BhdGgoYXJjaGl2ZSkuZXhwYW5kdXNlcigpXQogICAgZm9yIGEgaW4gYXJjaGl2ZXM6CiAgICAgICAgaWYgemlwZmlsZS5pc196aXBmaWxlKGEpIG9yIHRhcmZpbGUuaXNfdGFyZmlsZShhKToKICAgICAgICAgICAgZXh0cmFjdChhLCByYXcpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZGF0YXNldHMKCmRlZiB2ZXJpZnlfZGZkYyhyYXc6IFBhdGgpIC0+IGRpY3Q6CiAgICBkID0gZmluZF9kaXJfd2l0aChyYXcsICgibWV0YWRhdGEuanNvbiIsKSkKICAgIGlmIGQgaXMgTm9uZToKICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAicmVhc29uIjogIm1ldGFkYXRhLmpzb24gbm90IGZvdW5kIn0KICAgIHJldHVybiB7Im9rIjogVHJ1ZSwgInJvb3QiOiBzdHIoZCksICJ2aWRlb3MiOiBjb3VudChkLCAiKi5tcDQiKX0KCgpkZWYgZ2V0X2RmZGMocmF3OiBQYXRoLCBhcmNoaXZlOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gTm9uZToKICAgIGlmIGFyY2hpdmU6CiAgICAgICAgZXh0cmFjdChQYXRoKGFyY2hpdmUpLmV4cGFuZHVzZXIoKSwgcmF3LCBvbmx5X3ByZWZpeD0idHJhaW5fc2FtcGxlX3ZpZGVvcy8iKQogICAgICAgIHJldHVybgogICAgaWYgc2h1dGlsLndoaWNoKCJrYWdnbGUiKSBpcyBOb25lOgogICAgICAgIHN5cy5leGl0KCJrYWdnbGUgQ0xJIG5vdCBmb3VuZDogcGlwIGluc3RhbGwga2FnZ2xlIikKICAgIGlmIG5vdCAoUGF0aC5ob21lKCkgLyAiLmthZ2dsZSIgLyAia2FnZ2xlLmpzb24iKS5leGlzdHMoKSBhbmQgbm90IF9rYWdnbGVfZW52KCk6CiAgICAgICAgc3lzLmV4aXQoCiAgICAgICAgICAgICJLYWdnbGUgQVBJIHRva2VuIG1pc3NpbmcuIEthZ2dsZSAtPiBTZXR0aW5ncyAtPiBBUEkgLT4gJ0NyZWF0ZSBOZXcgVG9rZW4nLCB0aGVuIHB1dFxuIgogICAgICAgICAgICAia2FnZ2xlLmpzb24gaW4gfi8ua2FnZ2xlLyAoQ29sYWI6IHNlZSBub3RlYm9va3MvMDFfZG93bmxvYWRfZGF0YXNldHMuaXB5bmIpLlxuIgogICAgICAgICAgICAiQWxzbyBvcGVuIGh0dHBzOi8vd3d3LmthZ2dsZS5jb20vYy9kZWVwZmFrZS1kZXRlY3Rpb24tY2hhbGxlbmdlL3J1bGVzIGFuZCBhY2NlcHQgdGhlbS4iCiAgICAgICAgKQogICAgcmF3Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICMgVGhlIGRlZmF1bHQgY29tcGV0aXRpb24gZG93bmxvYWQgKH40IEdCKSBjb250YWlucyB0cmFpbl9zYW1wbGVfdmlkZW9zLyBhbmQgdGVzdF92aWRlb3MvLgogICAgc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgWyJrYWdnbGUiLCAiY29tcGV0aXRpb25zIiwgImRvd25sb2FkIiwgIi1jIiwgImRlZXBmYWtlLWRldGVjdGlvbi1jaGFsbGVuZ2UiLCAiLXAiLCBzdHIocmF3KV0sCiAgICAgICAgY2hlY2s9VHJ1ZSwKICAgICkKICAgIHppcF9wYXRoID0gcmF3IC8gImRlZXBmYWtlLWRldGVjdGlvbi1jaGFsbGVuZ2UuemlwIgogICAgZXh0cmFjdCh6aXBfcGF0aCwgcmF3LCBvbmx5X3ByZWZpeD0idHJhaW5fc2FtcGxlX3ZpZGVvcy8iKQogICAgemlwX3BhdGgudW5saW5rKCkKCgpkZWYgX2thZ2dsZV9lbnYoKSAtPiBib29sOgogICAgaW1wb3J0IG9zCgogICAgcmV0dXJuIGJvb2wob3MuZW52aXJvbi5nZXQoIktBR0dMRV9BUElfVE9LRU4iKSkgb3IgYm9vbCgKICAgICAgICBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX1VTRVJOQU1FIikgYW5kIG9zLmVudmlyb24uZ2V0KCJLQUdHTEVfS0VZIikKICAgICkKCgpkZWYgdmVyaWZ5X2FzdnNwb29mKHJhdzogUGF0aCkgLT4gZGljdDoKICAgIGQgPSBmaW5kX2Rpcl93aXRoKHJhdywgKCJBU1ZzcG9vZjIwMTlfTEFfY21fcHJvdG9jb2xzIiwgIkFTVnNwb29mMjAxOV9MQV90cmFpbiIpKQogICAgaWYgZCBpcyBOb25lOgogICAgICAgIHJldHVybiB7Im9rIjogRmFsc2UsICJyZWFzb24iOiAiTEEvIGZvbGRlciB3aXRoIHByb3RvY29scyBub3QgZm91bmQifQogICAgcmV0dXJuIHsKICAgICAgICAib2siOiBUcnVlLAogICAgICAgICJyb290Ijogc3RyKGQpLAogICAgICAgICoqe3M6IGNvdW50KGQgLyBmIkFTVnNwb29mMjAxOV9MQV97c30iLCAiKi5mbGFjIikgZm9yIHMgaW4gKCJ0cmFpbiIsICJkZXYiLCAiZXZhbCIpfSwKICAgIH0KCgpkZWYgZ2V0X2FzdnNwb29mKHJhdzogUGF0aCwgY2hlY2s6IGJvb2wgPSBGYWxzZSwga2VlcF96aXA6IGJvb2wgPSBGYWxzZSkgLT4gTm9uZToKICAgIGlmIGNoZWNrOgogICAgICAgIHdpdGggcmVxdWVzdHMuZ2V0KAogICAgICAgICAgICBBU1ZTUE9PRl9MQV9VUkwsCiAgICAgICAgICAgIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogVVNFUl9BR0VOVCwgIlJhbmdlIjogImJ5dGVzPTAtMyJ9LAogICAgICAgICAgICB0aW1lb3V0PTYwLAogICAgICAgICAgICBzdHJlYW09VHJ1ZSwKICAgICAgICApIGFzIHI6CiAgICAgICAgICAgIGhlYWRlciA9IG5leHQoci5pdGVyX2NvbnRlbnQoY2h1bmtfc2l6ZT00KSwgYiIiKQogICAgICAgICAgICBvayA9IHIuc3RhdHVzX2NvZGUgaW4gKDIwMCwgMjA2KSBhbmQgaGVhZGVyWzoyXSA9PSBiIlBLIgogICAgICAgICAgICBwcmludChmIkFTVnNwb29mIExBLnppcCBsaW5rOiBIVFRQIHtyLnN0YXR1c19jb2RlfSwgemlwIGhlYWRlciB7J09LJyBpZiBvayBlbHNlICdNSVNTSU5HJ30iKQogICAgICAgIHN5cy5leGl0KDAgaWYgb2sgZWxzZSAxKQogICAgdHJ5OgogICAgICAgIHppcF9wYXRoID0gaHR0cF9kb3dubG9hZChBU1ZTUE9PRl9MQV9VUkwsIHJhdyAvICJMQS56aXAiLCBleHBlY3RlZF9zaXplPUFTVlNQT09GX0xBX1NJWkUpCiAgICBleGNlcHQgcmVxdWVzdHMuSFRUUEVycm9yIGFzIGV4YzoKICAgICAgICBpZiBleGMucmVzcG9uc2UgaXMgTm9uZSBvciBleGMucmVzcG9uc2Uuc3RhdHVzX2NvZGUgIT0gNDAzOgogICAgICAgICAgICByYWlzZQogICAgICAgIHByaW50KCJFZGluYnVyZ2ggcmV0dXJuZWQgSFRUUCA0MDM7IHVzaW5nIHRoZSBBU1ZzcG9vZiBaZW5vZG8gbWlycm9yLiIpCiAgICAgICAgemlwX3BhdGggPSBodHRwX2Rvd25sb2FkKEFTVlNQT09GX0xBX01JUlJPUiwgcmF3IC8gIkxBLnppcCIsIGV4cGVjdGVkX3NpemU9QVNWU1BPT0ZfTEFfU0laRSkKICAgIGRpZ2VzdCA9IGhhc2hsaWIubWQ1KCkKICAgIHdpdGggemlwX3BhdGgub3BlbigicmIiKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogYXJjaGl2ZS5yZWFkKDEgPDwgMjApLCBiIiIpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQogICAgaWYgZGlnZXN0LmhleGRpZ2VzdCgpICE9IEFTVlNQT09GX0xBX01ENToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJBU1ZzcG9vZiBhcmNoaXZlIGNoZWNrc3VtIG1pc21hdGNoOiB7emlwX3BhdGh9OyBwcmVzZXJ2ZSBpdCBmb3IgaW5zcGVjdGlvbi4iKQogICAgZXh0cmFjdCh6aXBfcGF0aCwgcmF3KQogICAgaWYgbm90IGtlZXBfemlwOgogICAgICAgIHppcF9wYXRoLnVubGluaygpCgoKZGVmIHZlcmlmeV9mYWtlYXZjZWxlYihyYXc6IFBhdGgpIC0+IGRpY3Q6CiAgICBkID0gZmluZF9kaXJfd2l0aChyYXcsIEZBS0VBVkNFTEVCX0NBVEVHT1JJRVMpCiAgICBpZiBkIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsib2siOiBGYWxzZSwgInJlYXNvbiI6ICJjYXRlZ29yeSBmb2xkZXJzIChSZWFsVmlkZW8tUmVhbEF1ZGlvLCAuLi4pIG5vdCBmb3VuZCJ9CiAgICByZXR1cm4geyJvayI6IFRydWUsICJyb290Ijogc3RyKGQpLCAqKntjOiBjb3VudChkIC8gYywgIioubXA0IikgZm9yIGMgaW4gRkFLRUFWQ0VMRUJfQ0FURUdPUklFU319CgoKZGVmIGdldF9mYWtlYXZjZWxlYihyYXc6IFBhdGgsIHVybDogc3RyIHwgTm9uZSwgYXJjaGl2ZTogc3RyIHwgTm9uZSkgLT4gTm9uZToKICAgIGlmIG5vdCAodXJsIG9yIGFyY2hpdmUpOgogICAgICAgIHN5cy5leGl0KAogICAgICAgICAgICAiRmFrZUFWQ2VsZWIgcmVxdWlyZXMgYXBwcm92YWwuXG4iCiAgICAgICAgICAgIGYiICAxLiBGaWxsIGluIHRoZSByZXF1ZXN0IGZvcm06IHtGQUtFQVZDRUxFQl9GT1JNfVxuIgogICAgICAgICAgICAiICAyLiBZb3UnbGwgcmVjZWl2ZSBhIGRvd25sb2FkIGxpbmsvc2NyaXB0IGJ5IGVtYWlsLlxuIgogICAgICAgICAgICAiICAzLiBSZS1ydW4gd2l0aCAtLXVybCAnPGxpbms+JyBvciAtLWFyY2hpdmUgL3BhdGgvdG8vRmFrZUFWQ2VsZWJfdjEuMi56aXAiCiAgICAgICAgKQogICAgX2luZ2VzdChyYXcsIHVybCwgYXJjaGl2ZSkKCgpkZWYgdmVyaWZ5X2NlbGViZGYocmF3OiBQYXRoKSAtPiBkaWN0OgogICAgZCA9IGZpbmRfZGlyX3dpdGgocmF3LCBDRUxFQkRGX0ZPTERFUlMpCiAgICBpZiBkIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsib2siOiBGYWxzZSwgInJlYXNvbiI6ICJDZWxlYi1yZWFsIC8gQ2VsZWItc3ludGhlc2lzIC8gWW91VHViZS1yZWFsIG5vdCBmb3VuZCJ9CiAgICByZXR1cm4gewogICAgICAgICJvayI6IFRydWUsCiAgICAgICAgInJvb3QiOiBzdHIoZCksCiAgICAgICAgInRlc3RfbGlzdCI6IChkIC8gIkxpc3Rfb2ZfdGVzdGluZ192aWRlb3MudHh0IikuZXhpc3RzKCksCiAgICAgICAgKip7ZjogY291bnQoZCAvIGYsICIqLm1wNCIpIGZvciBmIGluIENFTEVCREZfRk9MREVSU30sCiAgICB9CgoKZGVmIGdldF9jZWxlYmRmKHJhdzogUGF0aCwgdXJsOiBzdHIgfCBOb25lLCBhcmNoaXZlOiBzdHIgfCBOb25lKSAtPiBOb25lOgogICAgaWYgbm90ICh1cmwgb3IgYXJjaGl2ZSk6CiAgICAgICAgc3lzLmV4aXQoCiAgICAgICAgICAgICJDZWxlYi1ERiB2MiByZXF1aXJlcyBhcHByb3ZhbC5cbiIKICAgICAgICAgICAgZiIgIDEuIFJlcXVlc3QgYWNjZXNzIHZpYSB0aGUgZm9ybSBsaW5rZWQgYXQge0NFTEVCREZfUkVQT31cbiIKICAgICAgICAgICAgIiAgMi4gUmUtcnVuIHdpdGggLS11cmwgJzxsaW5rPicgb3IgLS1hcmNoaXZlIC9wYXRoL3RvL0NlbGViLURGLXYyLnppcCIKICAgICAgICApCiAgICBfaW5nZXN0KHJhdywgdXJsLCBhcmNoaXZlKQoKClZFUklGWSA9IHsKICAgICJkZmRjIjogdmVyaWZ5X2RmZGMsCiAgICAiYXN2c3Bvb2YiOiB2ZXJpZnlfYXN2c3Bvb2YsCiAgICAiZmFrZWF2Y2VsZWIiOiB2ZXJpZnlfZmFrZWF2Y2VsZWIsCiAgICAiY2VsZWJkZiI6IHZlcmlmeV9jZWxlYmRmLAp9CgoKZGVmIHN0YXR1cyhjZmcpIC0+IE5vbmU6CiAgICBwcmludChmImRhdGFfcm9vdDoge2NmZ1snZGF0YV9yb290J119IikKICAgIGZvciBuYW1lLCBmbiBpbiBWRVJJRlkuaXRlbXMoKToKICAgICAgICBpbmZvID0gZm4oZ2V0X3BhdGgoY2ZnLCAicmF3IiwgbmFtZSkpCiAgICAgICAgZmxhZyA9ICJPSyAgICAgIiBpZiBpbmZvLnBvcCgib2siKSBlbHNlICJNSVNTSU5HIgogICAgICAgIHByaW50KGYiICBbe2ZsYWd9XSB7bmFtZToxMnN9IHtpbmZvfSIpCgoKZGVmIG1haW4oYXJndjogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIHAuYWRkX2FyZ3VtZW50KCJkYXRhc2V0IiwgY2hvaWNlcz1bInN0YXR1cyIsICpWRVJJRlldKQogICAgcC5hZGRfYXJndW1lbnQoIi0tdXJsIiwgaGVscD0iZG93bmxvYWQgbGluayByZWNlaXZlZCBieSBlbWFpbCAoZmFrZWF2Y2VsZWIgLyBjZWxlYmRmKSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1hcmNoaXZlIiwgaGVscD0iYWxyZWFkeS1kb3dubG9hZGVkIHppcC90YXIgdG8gZXh0cmFjdCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1jaGVjayIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9ImFzdnNwb29mOiBvbmx5IHRlc3QgdGhlIGRvd25sb2FkIGxpbmsiKQogICAgcC5hZGRfYXJndW1lbnQoIi0ta2VlcC16aXAiLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJrZWVwIGFyY2hpdmUgYWZ0ZXIgZXh0cmFjdGluZyIpCiAgICBhcmdzID0gcC5wYXJzZV9hcmdzKGFyZ3YpCgogICAgY2ZnID0gbG9hZF9jb25maWcoKQogICAgaWYgYXJncy5kYXRhc2V0ID09ICJzdGF0dXMiOgogICAgICAgIHN0YXR1cyhjZmcpCiAgICAgICAgcmV0dXJuCiAgICByYXcgPSBnZXRfcGF0aChjZmcsICJyYXciLCBhcmdzLmRhdGFzZXQpCiAgICBpZiBhcmdzLmRhdGFzZXQgPT0gImRmZGMiOgogICAgICAgIGdldF9kZmRjKHJhdywgYXJncy5hcmNoaXZlKQogICAgZWxpZiBhcmdzLmRhdGFzZXQgPT0gImFzdnNwb29mIjoKICAgICAgICBpZiBhcmdzLmFyY2hpdmU6CiAgICAgICAgICAgIGV4dHJhY3QoUGF0aChhcmdzLmFyY2hpdmUpLmV4cGFuZHVzZXIoKSwgcmF3KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGdldF9hc3ZzcG9vZihyYXcsIGNoZWNrPWFyZ3MuY2hlY2ssIGtlZXBfemlwPWFyZ3Mua2VlcF96aXApCiAgICBlbGlmIGFyZ3MuZGF0YXNldCA9PSAiZmFrZWF2Y2VsZWIiOgogICAgICAgIGdldF9mYWtlYXZjZWxlYihyYXcsIGFyZ3MudXJsLCBhcmdzLmFyY2hpdmUpCiAgICBlbGlmIGFyZ3MuZGF0YXNldCA9PSAiY2VsZWJkZiI6CiAgICAgICAgZ2V0X2NlbGViZGYocmF3LCBhcmdzLnVybCwgYXJncy5hcmNoaXZlKQogICAgaW5mbyA9IFZFUklGWVthcmdzLmRhdGFzZXRdKHJhdykKICAgIHByaW50KCgiVmVyaWZpZWQ6ICIgaWYgaW5mby5nZXQoIm9rIikgZWxzZSAiVmVyaWZpY2F0aW9uIEZBSUxFRDogIikgKyBzdHIoaW5mbykpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=")
(CODE / "src/deepfake/data/download.py").write_bytes(patch)
(logs / "download_patch_sha256.txt").write_text(hashlib.sha256(patch).hexdigest())
# Public real speech/spoof dataset; downloads resume after a disconnect.
run_module("deepfake.data.download", "asvspoof")
run_module("deepfake.data.manifest", "asvspoof")
run_module("deepfake.data.splits", "asvspoof")
run_module("deepfake.preprocessing.run", "asvspoof", "--per-class", PER_CLASS, "--workers", 2)
import pandas as pd
asv_index = get_path(cfg, "processed", "asvspoof", "index.csv")
asv = pd.read_csv(asv_index, keep_default_na=False)
display(asv.groupby(["split", "audio_label"]).size().unstack(fill_value=0))
errors = asv[asv.error != ""]
if len(errors):
    display(errors[["sample_id", "error"]].head(20))
    raise RuntimeError("Preprocessing errors must be resolved before training.")

In [ ]:
run_module("deepfake.training.train_audio", "--dataset", "asvspoof", "--run-name", AUDIO_RUN,
           "--epochs", 15, "--batch-size", 32, "--workers", 2, "--device", "cuda")
audio_dir = get_path(cfg, "checkpoints", AUDIO_RUN)
required = ["best.pt", "last.pt", "history.csv", "curves.png",
            "test/metrics.json", "test/predictions.csv", "test/per_category.csv"]
for name in required:
    assert (audio_dir / name).is_file(), f"Missing training output: {name}"
from IPython.display import Image, display
display(pd.read_csv(audio_dir / "history.csv"))
metrics = json.loads((audio_dir / "test/metrics.json").read_text())
print(json.dumps(metrics, indent=2))
display(Image(str(audio_dir / "curves.png")))
display(Image(str(audio_dir / "test/roc.png")))
display(pd.read_csv(audio_dir / "test/per_category.csv"))

In [ ]:
# Look only in the project's own folder for approved FakeAVCeleb archives/data.
from deepfake.data import download as dl
fake_index = get_path(cfg, "processed", "fakeavceleb", "index.csv")
if not fake_index.exists():
    archives = sorted(p for p in DATA.rglob("*.zip") if "fakeav" in p.name.lower())
    if archives:
        run_module("deepfake.data.download", "fakeavceleb", "--archive", archives[0])
    if dl.verify_fakeavceleb(get_path(cfg, "raw", "fakeavceleb"))["ok"]:
        run_module("deepfake.data.manifest", "fakeavceleb")
        run_module("deepfake.data.splits", "fakeavceleb")
        run_module("deepfake.preprocessing.run", "fakeavceleb", "--per-class", 400, "--workers", 2)
if fake_index.exists():
    visual_run = "visual_fakeavceleb_" + SOURCE_SHA256[:8]
    audio_ft = "audio_fakeavceleb_" + SOURCE_SHA256[:8]
    common = ["--dataset", "fakeavceleb", "--epochs", "10", "--workers", "2", "--device", "cuda"]
    run_module("deepfake.training.train_visual", *common, "--run-name", visual_run)
    run_module("deepfake.training.train_audio", *common, "--run-name", audio_ft,
               "--batch-size", 32, "--lr", "3e-5", "--freeze-epochs", 0,
               "--init-from", audio_dir / "best.pt")
    visual_ckpt = get_path(cfg, "checkpoints", visual_run, "best.pt")
    audio_ckpt = get_path(cfg, "checkpoints", audio_ft, "best.pt")
    models = ["--model", f"visual_only={visual_ckpt}", "--model", f"audio_only={audio_ckpt}"]
    for fusion in ["late", "intermediate"]:
        name = f"fusion_{fusion}_fakeavceleb_" + SOURCE_SHA256[:8]
        run_module("deepfake.training.train_fusion", *common, "--run-name", name,
                   "--fusion", fusion, "--visual-ckpt", visual_ckpt, "--audio-ckpt", audio_ckpt)
        models += ["--model", f"{fusion}={get_path(cfg, 'checkpoints', name, 'best.pt')}"]
    run_module("deepfake.evaluation.compare", "--dataset", "fakeavceleb", "--device", "cuda", *models)
    print("Multimodal training completed; inspect the saved validation/test results.")
else:
    print("MULTIMODAL PENDING: FakeAVCeleb is not available. The audio checkpoint remains saved.")
    print("Official access: https://github.com/DASH-Lab/FakeAVCeleb")

In [ ]:
run_module("deepfake.report", "--out", DATA / "report")
from IPython.display import Markdown
display(Markdown((DATA / "report/results.md").read_text()))
status = {"source_sha256": SOURCE_SHA256, "audio_checkpoint": str(audio_dir / "best.pt"),
          "audio_test_metrics": str(audio_dir / "test/metrics.json"),
          "multimodal_available": fake_index.exists(), "report": str(DATA / "report/results.md")}
(logs / "completion.json").write_text(json.dumps(status, indent=2))
print("Outputs saved persistently:", json.dumps(status, indent=2))